# Strain-Resolved Metagenomic, Functional, and Metabolic Modeling Measurements of Infant Gut Microbiome Assembly Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We demonstrate how to programmatically discover and analyze data based on Croissant schema entities referenced by their `@id`, ensuring transparency and reproducibility.

### Dataset Source
The dataset source is provided by a FAIR^2 Croissant schema at: [https://sen.science/doi/10.71728/senscience.49qh-764e/fair2.json](https://sen.science/doi/10.71728/senscience.49qh-764e/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We initialize the Dataset object and display key summary information from its metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.49qh-764e/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published on {metadata.datePublished}. Version: {metadata.version}\n")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We query the Croissant metadata for the list of record sets and display each along with their fields, referencing all entities by `@id`.

In [ ]:
# List all record sets with their @id
record_sets = metadata.recordSet
if record_sets and len(record_sets) > 0:
    print("RecordSets found:")
    for record_set in record_sets:
        print(f"  - RecordSet @id: {record_set['@id']}")
        # If available, list fields for each record set
        fields = record_set.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("    Fields:")
            for field in fields:
                fid = field.get('@id')
                name = field.get('name')
                dtype = field.get('dataType')
                print(f"      - Field @id: {fid}, name: {name}, dataType: {dtype}")
        else:
            print("    No fields listed.")
else:
    print("No record sets found in this dataset. Check the schema or the recordSet property.")

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview step. All extraction is referenced by `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = []

# Dynamically build record set list and extract data, referencing @id
if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        record_set_ids.append(rs_id)
        try:
            records = list(dataset.records(record_set=rs_id))
            if len(records) > 0:
                df = pd.DataFrame(records)
                dataframes[rs_id] = df
                print(f"Loaded DataFrame for RecordSet @id: {rs_id}")
                print(f"Columns: {df.columns.tolist()}")
                print(df.head(), "\n")
            else:
                print(f"No records found for RecordSet @id: {rs_id}")
        except Exception as e:
            print(f"Error loading records for RecordSet @id: {rs_id}: {e}")
else:
    print("No record sets defined. Unable to extract records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. The following example demonstrates numeric field filtering, normalization, and grouping by a categorical field—all referenced by field `@id` as per the Croissant schema. If the dataset does not provide numeric fields, adapt accordingly to available columns.

In [ ]:
# EDA: Example on the first available record set with numeric field
import numpy as np

chosen_rs_id = None
numeric_field_id = None
group_field_id = None
for rs in record_sets:
    rs_id = rs['@id']
    if rs_id in dataframes and not dataframes[rs_id].empty:
        # Find a numeric field
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            dtype = f.get('dataType')
            fid = f.get('@id')
            # Accept integers or floats
            if dtype and ("Float" in dtype or "Integer" in dtype or "Number" in dtype):
                if fid in dataframes[rs_id].columns:
                    numeric_field_id = fid
                    chosen_rs_id = rs_id
                    break
        # Pick a group field (categorical field)
        if fields and chosen_rs_id is not None:
            for f in fields:
                dtype = f.get('dataType')
                fid = f.get('@id')
                if dtype and ("Text" in dtype or "Categorical" in dtype):
                    if fid in dataframes[rs_id].columns:
                        group_field_id = fid
                        break
        if chosen_rs_id and numeric_field_id:
            break

# EDA processing
if chosen_rs_id and numeric_field_id:
    df = dataframes[chosen_rs_id]
    threshold = np.nanmean(df[numeric_field_id])
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records from RecordSet {chosen_rs_id} with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found in the available record sets for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: plot a histogram of the selected numeric field or a boxplot grouped by the categorical field if available.

In [ ]:
# Visualization: Histogram and boxplot
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id and numeric_field_id:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(data=dataframes[chosen_rs_id], x=numeric_field_id, ax=axes[0], kde=True)
    axes[0].set_title(f'Histogram of {numeric_field_id}')
    if group_field_id and group_field_id in dataframes[chosen_rs_id].columns:
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[chosen_rs_id], ax=axes[1])
        axes[1].set_title(f'Boxplot of {numeric_field_id} by {group_field_id}')
    else:
        axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No visualizable numeric field found.")

## 6. Conclusion
Through this notebook, we have demonstrated loading, overview, extraction, EDA, and visualization for a FAIR^2 dataset defined by a Croissant schema. Key entities (record sets, fields) were referenced by their `@id` throughout, as required for transparent and reproducible scientific data analysis.

Continue exploration by adapting field and record set selection, applying advanced analytics, and leveraging Croissant schema metadata for robust FAIR data workflows.